## Imports

In [2]:
import openai
from qdrant_client import QdrantClient
import dotenv

dotenv.load_dotenv()

True

## Embedding Function

In [3]:
def embed_text(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding




## Retrieval Function 

In [48]:
def retrieve_data(client, collection_name, query, embedding_model="text-embedding-3-small", top_k=5):

    query_vector = embed_text(query, embedding_model)


    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k
    )

    results = []
    for point in search_result.points:
        results.append({
            "id": point.id,
            "payload": point.payload,
            "score": point.score
        })

    return results

In [8]:
qdrant_host = "http://localhost:6333"
collection_name = "cm_interventions"

client = QdrantClient(url=qdrant_host)

query = "Pump vibration issue on machine HX-200 causing downtime"

results = retrieve_data(client, collection_name, query)

for idx, result in enumerate(results):
    print(f"Result {idx + 1}:")
    print(f"Qdrant ID: {result['id']}")
    print(f"Score: {result['score']}")
    print(f"Payload: {result['payload']}")
    print("-" * 40)

Result 1:
Qdrant ID: d6432b0f-44a3-457d-b289-746de4be32a2
Score: 0.6567029
Payload: {'id': 'INT-2024-0612', 'date_start': '2024-08-03 14:36', 'machine': 'HX-350', 'duration_min': 276, 'summary': '[FAULT_CODE] E-010\n[EVENT] Vibration monitoring system triggered E-010. Pump bearing vibration 880 mm/s RMS, above 3.5 mm/s threshold. Production continued at reduced rate pending investigation.\n[COMMENTS] Fault E-010: vibration 880 mm/s RMS at pump bearing. Spectrum analysis shows dominant 1× running speed component. Root cause: cable abrasion at conduit entry point from long-term vibration. Root cause isolated and corrected. Replaced associated consumable as preventive measure. Machine re-certified. Post-repair vibration: 0.3 mm/s RMS. Trending added to weekly checks. Machine returned to production. Monitoring interval increased for next 4 weeks.', 'embedding_model': 'text-embedding-3-small', 'created_at': '2026-03-28T11:20:38.230911'}
----------------------------------------
Result 2:
Qdr

## Format Context

In [24]:
def format_context(results):
    context = ""
    for result in results:
        payload = result['payload']
        context += f"ID: {payload.get('id', 'N/A')}\n"
        context += f"Machine: {payload.get('machine', 'N/A')}\n"
        context += f"Date: {payload.get('date_start', 'N/A')}\n"
        context += f"Summary: {payload.get('summary', 'N/A')}\n"
        context += "-" * 40 + "\n"
    return context

In [27]:
processed_context = format_context(results)
print("Formatted Context for LLM:", processed_context)

Formatted Context for LLM: ID: INT-2024-0612
Machine: HX-350
Date: 2024-08-03 14:36
Summary: [FAULT_CODE] E-010
[EVENT] Vibration monitoring system triggered E-010. Pump bearing vibration 880 mm/s RMS, above 3.5 mm/s threshold. Production continued at reduced rate pending investigation.
[COMMENTS] Fault E-010: vibration 880 mm/s RMS at pump bearing. Spectrum analysis shows dominant 1× running speed component. Root cause: cable abrasion at conduit entry point from long-term vibration. Root cause isolated and corrected. Replaced associated consumable as preventive measure. Machine re-certified. Post-repair vibration: 0.3 mm/s RMS. Trending added to weekly checks. Machine returned to production. Monitoring interval increased for next 4 weeks.
----------------------------------------
ID: INT-2023-1008
Machine: HX-200
Date: 2023-12-06 08:20
Summary: [FAULT_CODE] E-010
[RELATED_INTERVENTION] INT-2023-0974
[EVENT] Vibration monitoring system triggered E-010. Pump bearing vibration 2.1 mm/s RM

### Create Prompt Template

In [42]:
def build_prompt(context, query):
    prompt = f"""

    You are a maintenance assistant that can answer questions about past interventions, like possible root causes and actions for a given symptom.

    You will be given a question and a list of contexts.

    Instructions:
    - Use the contexts to answer the question.
    - If the context does not contain relevant information, say "I don't know".
    - Be concise and to the point.
    - Do not use markdown formatting.

    """

    prompt += f"\nQuestion: {query}\n\n"

    prompt += f"Contexts:\n{context}\n"

    return prompt

In [43]:
prompt = build_prompt(processed_context, query)
print("Constructed Prompt for LLM:", prompt)

Constructed Prompt for LLM: 

    You are a maintenance assistant that can answer questions about past interventions, like possible root causes and actions for a given symptom.

    You will be given a question and a list of contexts.

    Instructions:
    - Use the contexts to answer the question.
    - If the context does not contain relevant information, say "I don't know".
    - Be concise and to the point.
    - Do not use markdown formatting.

    
Question: Pump vibration issue on machine HX-200 causing downtime

Contexts:
ID: INT-2024-0612
Machine: HX-350
Date: 2024-08-03 14:36
Summary: [FAULT_CODE] E-010
[EVENT] Vibration monitoring system triggered E-010. Pump bearing vibration 880 mm/s RMS, above 3.5 mm/s threshold. Production continued at reduced rate pending investigation.
[COMMENTS] Fault E-010: vibration 880 mm/s RMS at pump bearing. Spectrum analysis shows dominant 1× running speed component. Root cause: cable abrasion at conduit entry point from long-term vibration. R

## Define the Generation

In [44]:
def generate_answer(prompt, generation_model="gpt-5.4-nano"):
    response = openai.chat.completions.create(
        model=generation_model,
        messages=[
            {"role": "system", "content": prompt}
            ],
        reasoning_effort="low"
    )
    return response.choices[0].message.content.strip()

In [45]:
answer = generate_answer(prompt)
print("Generated Answer from LLM:", answer)

Generated Answer from LLM: For machine HX-200 (FAULT_CODE E-010 on 2023-12-06), the pump vibration issue was traced to drive belt elongation.

What was observed
- Vibration monitoring system triggered E-010 at the pump bearing.
- Spectrum analysis showed a dominant 1× running speed component (typical of drive/power transmission issues).
- Measured pump bearing vibration: 2.1 mm/s RMS (system threshold stated as 3.5 mm/s RMS in the record).
- Production continued at reduced rate pending investigation.

Root cause
- Drive belt elongation beyond the tensioner adjustment range.

Actions taken
- Full LOTO.
- Replaced the affected component (drive belt).
- Checked for secondary damage across related systems; none found.
- Added/continued weekly vibration trending after the repair.
- Inspected related components as a precaution and found them normal.
- Post-repair vibration reported as 0.3 mm/s RMS (and machine returned to production per the intervention record pattern).


## Combine RAG Pipeline

In [52]:
def rag_pipeline(client, collection_name, query, embedding_model="text-embedding-3-small", generation_model="gpt-5.4-nano", top_k=5):
    results = retrieve_data(client, collection_name, query, embedding_model, top_k)
    context = format_context(results)
    prompt = build_prompt(context, query)
    answer = generate_answer(prompt, generation_model)
    return answer

In [55]:
query = "Pump vibration issue on machine HX-200 causing downtime"

final_answer = rag_pipeline(client, collection_name, query, top_k=5)
print("Final Answer from RAG Pipeline:", final_answer)

Final Answer from RAG Pipeline: For machine HX-200, the past pump vibration issue was logged as fault E-010 (vibration monitoring system triggered) on 2023-12-06 08:20.

- Observed symptom: Pump bearing vibration 2.1 mm/s RMS (reported as above the 3.5 mm/s threshold while production continued at reduced rate pending investigation).
- Spectrum result: Dominant 1× running speed component.
- Root cause: Drive belt elongation beyond the tensioner adjustment range.
- Actions taken:
  1) Full LOTO performed
  2) Component replaced (drive belt)
  3) Checked related systems for secondary damage (none found)
  4) Added trending to weekly checks and inspected related components as a precaution
- Post-repair result: Vibration reduced to 0.3 mm/s RMS (reported as “210 mm/s RMS” in the record, but the follow-up statement indicates a successful low reading; if you want I can reconcile this discrepancy against your historian values).
- I don’t know: the specific downtime duration/cause on HX-200 for